In [1]:
import yaml

with open("../configs/pretrain_project/silica/baselines/config_cgcnn.yml") as f:
# with open("../configs/pretrain_project/silica/baselines/config_torchmd.yml") as f:
# with open("../configs/pretrain_project/silica/baselines/config_pbc_processed.yml") as f:
# with open("../configs/pretrain_project/silica/baselines/config_pbc_processed_as.yml") as f:
    config = yaml.safe_load(f)

In [10]:
import os, torch

os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

In [2]:
from matdeeplearn.trainers.base_trainer import BaseTrainer

dataset = BaseTrainer._load_dataset(config["dataset"], config["task"]["run_mode"]) if "src" in config["dataset"] else None
model1 = BaseTrainer._load_model(config["model"], config["dataset"]["preprocess_params"], dataset, 1, 0)[0]
model2 = BaseTrainer._load_model(config["model"], config["dataset"]["preprocess_params"], dataset, 1, 0)[0]
sampler = BaseTrainer._load_sampler(config["optim"], dataset, 1, 0) if "src" in config["dataset"] else None

/net/csefiles/coc-fung-cluster/Qianyu/stable_md/MatDeepLearn_dev/matdeeplearn/preprocessor/datasets.py:25: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slic

In [18]:
import torch

# checkpoint_pth = "../results/cgcnn/2024-06-14-09-41-25-357-cgcnn_sio2/checkpoint_0/best_checkpoint.pt"
# checkpoint_pth = "../results/2024-09-18-22-36-39-925-silica_torchmd/checkpoint_0/best_checkpoint.pt"
checkpoint_pth = "../results/2024-10-26-23-56-08-189-graphormer3d_pbc_gbf/checkpoint_0/best_checkpoint.pt"
# checkpoint_pth = "../results/2024-11-03-16-24-35-739-graphormer3d_pbc_gbf_as_no_force_head/checkpoint_0/best_checkpoint.pt"

model1.load_state_dict(torch.load(checkpoint_pth, map_location="cpu")["state_dict"])
model2.load_state_dict(torch.load(checkpoint_pth, map_location="cpu")["state_dict"])

/tmp/ipykernel_3284705/3363203874.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model1.load_state_dict(torch.load(checkpoint_pth, map_location="cpu")["state_dict"])
/t

<All keys matched successfully>

In [3]:
from torch_geometric.loader import DataLoader
from matdeeplearn.preprocessor.pbc_transform import Batch

loader = DataLoader(dataset['test'], batch_size=1, shuffle=False, sampler=sampler)
# loader.collate_fn = Batch.from_datalist

In [20]:
import torch.nn.functional as F
from matdeeplearn.modules.loss import ForceLoss

loss = ForceLoss(weight_energy=0.01, weight_force=50.0)
model1 = model1.to("cuda")
model1.eval()

energy_losses = []
force_losses = []
with torch.no_grad():
    for data in loader:
        data = data.to("cuda")
        out = model1(data)
        e_loss = F.l1_loss(out["output"], data.y)
        f_loss = F.l1_loss(out["pos_grad"], data.forces)
        energy_losses.append(e_loss.item())
        force_losses.append(f_loss.item())

avg_energy_loss = sum(energy_losses) / len(energy_losses)
avg_force_loss = sum(force_losses) / len(force_losses)

print(f"Energy loss: {avg_energy_loss:.5f}, scaled: {0.01 * avg_energy_loss:.5f}")
print(f"Force loss: {avg_force_loss:.5f}, scaled: {50 * avg_force_loss:.5f}")
print(f"Scaled total loss: {0.01 * avg_energy_loss + 50 * avg_force_loss:.5f}")

Energy loss: 2.19190, scaled: 0.02192
Force loss: 0.03070, scaled: 1.53490
Scaled total loss: 1.55682


In [21]:
import torch

for param in model1.parameters():
    param.data = param.data.to(torch.bfloat16)

In [ ]:
from torchao.quantization import quantize_, int4_weight_only
group_size = 32

# you can enable [hqq](https://ithub.com/mobiusml/hqq/tree/master) quantization which is expected to improves accuracy through
# use_hqq flag for `int4_weight_only` quantization
use_hqq = False
quantize_(model1, int4_weight_only(group_size=group_size, use_hqq=use_hqq))

In [4]:
import numpy as np
import torch

model = model1.to("cuda:0")
model.eval()

starter, ender = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
repetitions = 50
timings = np.zeros((repetitions,1))

#GPU-WARM-UP
for i in range(50):
    batch = next(iter(loader)).to("cuda:0")
    _ = model(batch)

# MEASURE PERFORMANCE
with torch.no_grad():
    for rep in range(repetitions):
        starter.record()
        for batch in loader:
            batch = batch.to("cuda:0")
            _ = model(batch)
        ender.record()
        # WAIT FOR GPU SYNC
        torch.cuda.synchronize()
        curr_time = starter.elapsed_time(ender)
        timings[rep] = curr_time

mean_syn = np.sum(timings) / repetitions
std_syn = np.std(timings)
print(f"Mean inference time: {mean_syn:.2f} ms")
print(f"Std  inference time: {std_syn:.2f} ms")

Mean inference time: 1456.97 ms
Std  inference time: 581.05 ms


In [6]:
import torch.nn.functional as F
from matdeeplearn.modules.loss import ForceLoss

loss = ForceLoss(weight_energy=0.01, weight_force=50.0)
model1 = model1.to("cuda")
model1.eval()

energy_losses = []
force_losses = []
with torch.no_grad():
    for data in loader:
        data = data.to("cuda")
        out = model1(data)
        e_loss = F.l1_loss(out["output"], data.y)
        f_loss = F.l1_loss(out["pos_grad"], data.forces)
        energy_losses.append(e_loss.item())
        force_losses.append(f_loss.item())

avg_energy_loss = sum(energy_losses) / len(energy_losses)
avg_force_loss = sum(force_losses) / len(force_losses)

print(f"Energy loss: {avg_energy_loss:.5f}, scaled: {0.01 * avg_energy_loss:.5f}")
print(f"Force loss: {avg_force_loss:.5f}, scaled: {50 * avg_force_loss:.5f}")
print(f"Scaled total loss: {0.01 * avg_energy_loss + 50 * avg_force_loss:.5f}")

Energy loss: 2.88693, scaled: 0.02887
Force loss: 0.09861, scaled: 4.93061
Scaled total loss: 4.95948
